In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow as pa
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print('Libraries loaded')

print("Loading data")
cols = [
'dti',
'fico_range_low',
'inq_last_6mths',
'int_rate',
'issue_d',
'loan_amnt',
'loan_status',
'pub_rec',
'revol_util',
'sub_grade',
'term'
]
dfa_data = pd.read_parquet('LCA_cleaned_final', columns=cols) #2,260,701 

dlqcy = ['Charged Off','Late (31-120 days)','Late (16-30 days)','Default']
dflt = ['Charged Off','Default']

dfa_data['delinquency_tf'] = dfa_data['loan_status'].isin(dlqcy).astype(int)
dfa_data['defaulted_tf'] = dfa_data['loan_status'].isin(dflt).astype(int)

dfa = dfa_data.sample(n=500000, random_state=42) #Will contain the columns above, plus the new columns for delinquency and default. 

print("Complete")

Libraries loaded
Loading data
Complete


In [2]:
# Target is 'defaulted_tf' or 'delinquency_tf'

default_predictors = ['sub_grade','revol_util','inq_last_6mths', 'loan_amnt', 'int_rate','term']   # your chosen predictors

X = dfa[default_predictors].copy()   # Predictor matrix... .copy() is used to avoid SettingWithCopyWarning. It creates a new DataFrame that is a copy of the original. So that any changes made to X do not affect dfa. This is important for data integrity and avoiding unintended side effects in your analysis.
y = dfa['defaulted_tf']              # Target vector

In [3]:
X = pd.get_dummies(X, columns=['sub_grade'], drop_first=True) # One-hot encode the 'sub_grade' categorical into a matrix of binary features. 
#Note - Can only run this once, as it will create new columns in X. If you run it again, it will throw an error because the columns already exist.

In [4]:
print(X.head(8).to_string())

                     revol_util  inq_last_6mths  loan_amnt  int_rate  term  sub_grade_A2  sub_grade_A3  sub_grade_A4  sub_grade_A5  sub_grade_B1  sub_grade_B2  sub_grade_B3  sub_grade_B4  sub_grade_B5  sub_grade_C1  sub_grade_C2  sub_grade_C3  sub_grade_C4  sub_grade_C5  sub_grade_D1  sub_grade_D2  sub_grade_D3  sub_grade_D4  sub_grade_D5  sub_grade_E1  sub_grade_E2  sub_grade_E3  sub_grade_E4  sub_grade_E5  sub_grade_F1  sub_grade_F2  sub_grade_F3  sub_grade_F4  sub_grade_F5  sub_grade_G1  sub_grade_G2  sub_grade_G3  sub_grade_G4  sub_grade_G5
__null_dask_index__                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,                    # X and y will be split into training and testing sets
    test_size=0.3,           # 30% held out for testing
    random_state=42,         # Makes results reproducible across runs
    stratify=y               # Preserve class balance (important for rare events like default)
)

In [ ]:
model = RandomForestClassifier(
    n_estimators=500,                              
    max_depth=14,                               # Max depth is used for regularization to prevent overfitting. Regularization is important for rare events like default, as the model can easily overfit to the majority class.
    random_state=42,                               
    n_jobs=-1,                              
    class_weight='balanced'                      #Class weight is used to give more importance to the minority class (defaulted loans) during training. Choices are 'balanced', 'balanced_subsample', or a dictionary of class weights. 'balanced' automatically adjusts weights inversely proportional to class frequencies in the input data.
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Get probability of positive class


y_pred_custom = (y_pred_proba >=  0.5).astype(int) #Threshold value for classification. 


print("Classification Report:")
print(classification_report(y_test, y_pred))
#print("\nCustom Classification Report:")
#print(classification_report(y_test, y_pred_custom))
print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 4))


In [7]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop Features:")
print(importance)


Top Features:
           feature  importance
3         int_rate    0.454503
0       revol_util    0.137780
2        loan_amnt    0.105650
1   inq_last_6mths    0.061598
4             term    0.040956
7     sub_grade_A4    0.023775
5     sub_grade_A2    0.021464
6     sub_grade_A3    0.020254
8     sub_grade_A5    0.012859
9     sub_grade_B1    0.012594
14    sub_grade_C1    0.010929
13    sub_grade_B5    0.010816
10    sub_grade_B2    0.009530
11    sub_grade_B3    0.007436
12    sub_grade_B4    0.007384
18    sub_grade_C5    0.007350
19    sub_grade_D1    0.007317
25    sub_grade_E2    0.005240
26    sub_grade_E3    0.004734
15    sub_grade_C2    0.003835
24    sub_grade_E1    0.003805
16    sub_grade_C3    0.003466
20    sub_grade_D2    0.003438
27    sub_grade_E4    0.003356
17    sub_grade_C4    0.003156
22    sub_grade_D4    0.002975
21    sub_grade_D3    0.002642
23    sub_grade_D5    0.002511
29    sub_grade_F1    0.001615
28    sub_grade_E5    0.001424
30    sub_grade_F2    0.

In [8]:
#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)


Confusion Matrix:
[[84885 47280]
 [ 5557 12278]]


In [9]:
print("Iteration note: Max depth changed to 14, test size still 0.5, n_estimators still 500.")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 4))
print("\nConfusion Matrix:")
print(cm)
fp_mask = (y_test == 0) & (y_pred == 1)

subgrade_cols = [col for col in X_test.columns if col.startswith('sub_grade_')]
fp_subgrades = X_test.loc[fp_mask, subgrade_cols].idxmax(axis=1).str.replace('sub_grade_', '')

if len(fp_subgrades) > 0:
    fp_composition = fp_subgrades.value_counts(normalize=True) * 100
    print("\nFalse Positive Cohort - Sub-grade Composition (%):")
    print(fp_composition.round(2).sort_index())
else:
    print("\nNo false positives in this run.")

Iteration note: Max depth changed to 14, test size still 0.5, n_estimators still 500.
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.64      0.76    132165
           1       0.21      0.69      0.32     17835

    accuracy                           0.65    150000
   macro avg       0.57      0.67      0.54    150000
weighted avg       0.85      0.65      0.71    150000


ROC-AUC Score: 0.7262

Confusion Matrix:
[[84885 47280]
 [ 5557 12278]]

False Positive Cohort - Sub-grade Composition (%):
B3     0.01
B4     1.14
B5     2.50
C1     5.34
C2     9.12
C3     9.20
C4    12.17
C5    10.44
D1     7.86
D2     7.36
D3     6.95
D4     5.54
D5     5.08
E1     3.02
E2     2.65
E3     2.42
E4     2.18
E5     2.21
F1     1.24
F2     0.80
F3     0.74
F4     0.54
F5     0.43
G1     0.40
G2     0.22
G3     0.17
G4     0.12
G5     0.14
Name: proportion, dtype: float64


## Model Development Log

### 7/9/2026 – Initial Modeling Results

**Pass 1** (no class weighting)  
Using a 30% test set (~150k rows), the model predicted **zero** Class 1 (default) outcomes.

                precision    recall  f1-score   support
    0               0.88      1.00      0.94    132165
    1               0.00      0.00      0.00     17835
    accuracy                            0.88    150000
    macro avg       0.44      0.50      0.47    150000
    weighted avg    0.78      0.88      0.83    150000
    ROC-AUC: 0.71

**Pass 2** (class_weight='balanced')  
Switching to balanced class weights produced a large improvement in recall on the minority class:

                precision    recall  f1-score   support
    0               0.94      0.58      0.71    132165
    1               0.19      0.72      0.30     17835
    accuracy                            0.59    150000
    macro avg       0.56      0.65      0.51    150000
    weighted avg    0.85      0.59      0.67    150000
    ROC-AUC: 0.7108


### Interpreting the False Positives

Even though many of the model’s positive predictions are technically “false positives” relative to Lending Club’s final outcomes, this does not necessarily mean the model is wrong. Lending Club’s historical approval decisions, intuitively, seem off to me — they are the decisions of a company that ultimately struggled with elevated charge-offs.

**False Positive cohort characteristics** (approx. 55k loans):
- Highest sub-grade present: B4
- ~45% of the cohort is D1 or worse
- Average revolving utilization: 57.1%
- Average inquiries in last 6 months: 0.80 (most loans had at least one inquiry)
- Total loan volume in the cohort: ~$872 million
- Average interest rate: ~17%

In the False Positive cohort: It is **absolutely NOT the case** that the majority of the loans were 'performing just fine'. The breakdown of that cohort is that they simply did not hit the specific 'default' or 'charged off' status based on Lending Club's business decisions. 

#### 7/18/2026 Update

I reviewed the one-hot encoded False Positive cohort in detail. When I first shared the results, the initial LLM response treated the large FP volume as straightforward model error and pushed back on the idea that many of these loans should have been viewed as higher risk. After going back and forth — and insisting on looking at the actual risk characteristics inside the FP cohort - I turned out to be correct. The above MLA is performing well **in spite of false positives.** The FP data - contains loans that **should have** defaulted or charged off. 

Additional checks:
- Adjusting the decision threshold away from the default 0.5 did not improve ROC-AUC. 0.5 remains the most effective threshold.
- Increasing the sample size from 150k to 450k left ROC-AUC essentially unchanged. Performance is stable.

#### 7/19/2026 Updates

Scaled testing to larger samples (up to ~990k rows). ROC-AUC stayed consistent in the ~0.72 range, with only minor gains in F1. The model’s performance does not degrade or inflate artificially with sample size.

Interaction terms (`int_rate * dti`, `int_rate * loan_amnt`, `dti * loan_amnt`) were tested and did not produce meaningful improvements in precision or overall metrics.

Hyperparameter exploration:
- Increasing `n_estimators` and `max_depth` generally improved balance between classes.
- Lowering `max_depth` increased true positives (more defaults caught) but also drove up false positives substantially.
- Current best trade-off sits around `max_depth=14–16`.

Further tuning of `n_estimators`, `max_depth`, and test-set size continues to raise true positives, but at the cost of higher false positives. The classification report becomes more balanced overall, yet the FP count keeps climbing.

Why the persistent false positives?
- They are heavily concentrated in C and D sub-grades (with some E), which matches the composition of Lending Club’s overall book.
- Nearly 40% of the portfolio sat in C–G grades. After the 2008 crisis, Lending Club continued an aggressive sub-prime strategy, relying primarily on higher interest rates to offset expected losses rather than tightening underwriting standards.
- Given that portfolio mix, further large gains in true-positive rate may be difficult without accepting even higher false-positive volume. The current model appears to be performing about as well as the underlying risk distribution allows.

In [10]:
overall_subgrade = dfa_data['sub_grade'].value_counts(normalize=True) * 100

print("Overall Sub-grade Composition in Full Dataset (%):")
print(overall_subgrade.round(2).sort_index())

Overall Sub-grade Composition in Full Dataset (%):
sub_grade
A1    3.84
A2    3.08
A3    3.24
A4    4.24
A5    4.76
B1    5.54
B2     5.6
B3    5.82
B4    6.18
B5    6.21
C1    6.45
C2     5.8
C3    5.71
C4    5.62
C5    5.16
D1    3.62
D2    3.22
D3    2.87
D4    2.52
D5    2.12
E1    1.49
E2    1.32
E3    1.18
E4    1.01
E5     1.0
F1    0.59
F2    0.41
F3    0.34
F4    0.27
F5    0.23
G1    0.18
G2    0.12
G3    0.09
G4    0.08
G5    0.07
Name: proportion, dtype: Float64


In [11]:
P = dfa_data[default_predictors].copy()   
t = dfa_data['defaulted_tf']

In [12]:
P = pd.get_dummies(P, columns=['sub_grade'], drop_first=True)

In [17]:
# Final model on full dataset

final_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=14,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

final_model.fit(P, t)

full_pred_proba = final_model.predict_proba(P)[:, 1]
full_pred = (full_pred_proba >= 0.5).astype(int)

# Overall predicted default rate
predicted_default_rate = full_pred.mean() * 100

print(f"Overall Predicted Default Rate on Full Dataset: {predicted_default_rate:.2f}%")
print(f"Total predicted defaults: {full_pred.sum():,}")
print(f"Total loans: {len(full_pred):,}")

Overall Predicted Default Rate on Full Dataset: 42.71%
Total predicted defaults: 965,451
Total loans: 2,260,701


In [14]:
# Breakdown by key cohorts
dfa_data['predicted_default'] = full_pred

print("Predicted Default Rate by Sub-Grade:")
print(dfa_data.groupby('sub_grade')['predicted_default'].mean().round(4) * 100)

print("\nPredicted Default Rate by Issue Year:")
print(dfa_data.groupby(dfa_data['issue_d'].dt.year)['predicted_default'].mean().round(4) * 100)

Predicted Default Rate by Sub-Grade:
sub_grade
A1     0.00
A2     0.08
A3     0.11
A4     0.17
A5     0.20
B1     0.24
B2     0.43
B3     1.76
B4     6.40
B5    21.79
C1    34.36
C2    50.65
C3    56.57
C4    67.51
C5    64.35
D1    76.94
D2    82.18
D3    79.81
D4    83.75
D5    83.98
E1    86.59
E2    87.76
E3    86.51
E4    87.29
E5    93.34
F1    95.39
F2    94.62
F3    97.18
F4    97.24
F5    97.99
G1    98.68
G2    97.92
G3    99.38
G4    99.36
G5    98.41
Name: predicted_default, dtype: float64

Predicted Default Rate by Issue Year:
issue_d
2007.0    14.43
2008.0    17.93
2009.0    32.97
2010.0    32.94
2011.0    35.58
2012.0    42.67
2013.0    46.24
2014.0    51.42
2015.0    54.93
2016.0    49.48
2017.0    28.26
2018.0     7.25
Name: predicted_default, dtype: float64


In [15]:
print("Total rows in dfa_data:", len(dfa_data))
print("Unique issue years:", dfa_data['issue_d'].dt.year.nunique())

Total rows in dfa_data: 2260701
Unique issue years: 12


In [16]:
year_comparison = (
    dfa_data
    .groupby(dfa_data['issue_d'].dt.year)
    .agg(
        actual_default_rate=('defaulted_tf', 'mean'),
        predicted_default_rate=('predicted_default', 'mean')
    )
    .round(4) * 100
)
year_comparison['difference'] = year_comparison['predicted_default_rate'] - year_comparison['actual_default_rate']
year_comparison['total_loans'] = dfa_data.groupby(dfa_data['issue_d'].dt.year).size()

print("Predicted vs Actual Default Rate by Issue Year (%):")
print(year_comparison)

Predicted vs Actual Default Rate by Issue Year (%):
         actual_default_rate  predicted_default_rate  difference  total_loans
issue_d                                                                      
2007.0                  7.46                   14.43        6.97          603
2008.0                 10.32                   17.93        7.61         2393
2009.0                 11.25                   32.97       21.72         5281
2010.0                 11.86                   32.94       21.08        12537
2011.0                 15.18                   35.58       20.40        21721
2012.0                 16.20                   42.67       26.47        53367
2013.0                 15.59                   46.24       30.65       134814
2014.0                 17.47                   51.42       33.95       235629
2015.0                 18.00                   54.93       36.93       421095
2016.0                 15.71                   49.48       33.77       434407
2017.0      

## Conclusion (So far)

As my first ML project I built a Random Forest classifier to examine default risk in Lending Club’s 2007–2018 loan data.

The strongest, most defensible finding is the clear monotonic relationship between sub-grade and predicted default probability. Risk rises sharply once loans move into the C–G range. That ranking held up consistently and matches the concentration of higher-risk loans already visible in the exploratory analysis.

The year-over-year pattern is also useful. Predicted risk rises through the 2013–2015 growth years and then falls in 2017–2018 after Lending Club tightened. The *shape* of that gap tracks both the internal data and external reporting about their underwriting at the time. The absolute size of the gap should be read carefully — class weighting and in-sample evaluation both inflate the predicted rates — but the pattern itself still adds support to the broader story.

Traditional banks are generally required to charge off closed-end consumer loans at 120 days past due so non-performing loans don’t remain on their books as assets. During the period of this dataset, Lending Club was a marketplace lender (not a bank), so it was not subject to the same rigid charge-off timeline and had more flexibility in how long loans could sit in late or delinquent status before being formally charged off. That difference in constraints is relevant when interpreting the model’s higher risk flags.

Overall, the work supports the view that Lending Club carried elevated risk in the mid-2010s relative to a more conservative approach, and the model’s higher risk flags are directionally consistent with that assessment.

### Next steps that would tighten the results
- Re-run the year and sub-grade comparisons on a true held-out test set
- Compare the full `loan_status` breakdown of false positives vs true negatives
- Confirm no post-origination leakage features remain in the model

### A note on the dataset

The Lending Club 2007–2018 dataset is widely used for educational projects precisely because it is messy in realistic ways. It has severe class imbalance, ambiguous intermediate loan statuses (Current, Late, In Grace Period), potential target leakage from post-origination fields, and a large volume of loans that force careful decisions about sampling and memory. Working through those issues — rather than starting with a clean, balanced toy dataset — is what makes the project useful for building practical judgment.

## Outside Sources

On **July 9, 2026** — while still in the middle of analysis and before looking at any external articles — I wrote down an attempted prediction in my PROJECT_PROGRESS.md file about what I thought was wrong with Lending Club’s underwriting.

My hypothesis at the time was that the model’s “gap” (especially the higher predicted default rates on lower-grade loans) was not a failure. It was pointing to a real weakness: Lending Club had been approving higher-risk loans (D-grade and below) more aggressively than the risk justified, prioritizing volume over tighter controls.

I deliberately avoided reading articles, forums, or industry commentary while working on the project so I would not bias my own results. Only after finishing the analysis did I go look for external confirmation.

Here is what other sources reported after the fact, and how it lines up with the prediction I made on July 9th:

https://www.pymnts.com/news/alternative-financial-services/2016/lendingclubs-skyrocketing-charge-offs/
- “Charge-offs … are up 38 percent since 2013 for LendingClub.”
- “LendingClub’s lower-graded loans saw gross charge-offs pick up 6.31 percent between 2013 and 2015. Charge-off rates on top-graded loans … rose less dramatically, to 1.51 percent from 1.46 percent.”

https://www.morningstar.com/stocks/market-gives-lendingclub-too-much-credit
- “We have concerns that LendingClub will be encouraged to continually loosen its credit standards to drive growth.”
- “In 2011, a borrower with a FICO score of 715 would probably have been rated B2, whereas in 2014, a similar FICO score of 715 was likely to be rated three subgrades higher at A4.”

https://www.reuters.com/article/lendingclub-loans-idUSL1N18Z1MG/
- “Lending Club Corp said … it was cutting back loans to riskier borrowers and raising interest rates…”
- “It expects its standard loan volume to decrease by around 5 percent due to tightened credit criteria for borrowers.”

https://techcrunch.com/2017/03/28/tech-will-lead-to-new-sub-prime-crunch/
- “Delinquencies are growing, especially when it comes to high-risk loans.”

https://www.crowdfundinsider.com/2016/06/86590-lending-club-raises-interest-rates-updates-on-loan-programs/
- “Interest rates will increase by a weighted average of 55 basis points … Rates are increasing across all grades but changes are concentrated in grades D, E and F.”
- “Debt to income criteria has been reduced to 35% from 40% … This change is expected to impact loans grade E through G with standard loan volumes being reduced by 5%.”

https://finainews.com/lending/lending-club-underwriting-questioned-as-chargeoffs-climb/ 
- “Chargeoff rates at Lending Club are up 38% since 2013.”
- “Gross chargeoffs on Lending Club’s lower-rated loans one year after issuance reached 6.3% in 2016, up from 4.6% in 2013.”





# Notes for later:

- Create an 'interaction feature' by multiplying two (or more?) numeric columns. 

In [ ]:
results = X_test.copy()
results['actual'] = y_test.values
results['predicted'] = y_pred
results['proba_default'] = y_pred_proba

# Create category column
def get_category(row):
    if row['actual'] == 0 and row['predicted'] == 0:
        return 'TN'
    elif row['actual'] == 0 and row['predicted'] == 1:
        return 'FP'
    elif row['actual'] == 1 and row['predicted'] == 0:
        return 'FN'
    else:
        return 'TP'

results['category'] = results.apply(get_category, axis=1)

print("Results dataframe created successfully.")
print(results['category'].value_counts())


results[results['category'] == 'TP'].to_excel(r'C:\Users\spenc\Downloads\True_Positives.xlsx', index=False)
results[results['category'] == 'FP'].to_excel(r'C:\Users\spenc\Downloads\False_Positives.xlsx', index=False)
results[results['category'] == 'FN'].to_excel(r'C:\Users\spenc\Downloads\False_Negatives.xlsx', index=False)
results[results['category'] == 'TN'].to_excel(r'C:\Users\spenc\Downloads\True_Negatives.xlsx', index=False)
print("\n Files exported")

In [ ]:
# =============================================================================
# Template for a basic machine learning workflow in Python
# =============================================================================


"""
STEP 1: Load the cleaned data
- Use the parquet file from File 1 (much faster than CSV)
- In finance projects, always prefer Parquet for large datasets
"""


"""
STEP 2: Define target and features
- Target = binary outcome we want to predict (default / delinquency)
- Features = the variables we selected after EDA, VIF, and importance checks
- Repeatable tip: Keep a clear list here so you can easily swap models later
"""
target = '<your_target_column>'          # e.g. 'defaulted_tf' or 'delinquency_tf'

features = ['<column1>', '<column2>', '<column3>']   # your chosen predictors

y = dfa[target]            # Target vector. A series of 0s and 1s that the model is learning to predict. 
X = dfa[features].copy()   # Predictor matrix. A pandas DataFrame containing the features used to predict the above target.


"""
STEP 3: Train / Test Split
- Always split before training so you can honestly test generalization
- stratify=y keeps the proportion of good/bad loans similar in both sets
- In finance, this is critical because class imbalance (few defaults) is common
"""
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3,           # 30% held out for testing
    random_state=42,         # Makes results reproducible across runs
    stratify=y               # Preserve class balance (important for rare events like default)
)

"""
STEP 4: Train the model
- Random Forest is a good starting point for tabular finance data
- n_estimators = number of trees (more = more stable but slower)
- max_depth = prevents trees from growing too deep (helps control overfitting)
"""
model = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1                # Use all CPU cores for speed
)
model.fit(X_train, y_train)

"""
STEP 5: Evaluate performance
- classification_report gives precision, recall, F1 per class
- ROC-AUC is especially useful in finance (measures ability to rank risk)
"""
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]   # Probability of positive class (default)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:", round(roc_auc_score(y_test, y_pred_proba), 4))

"""
STEP 6: Feature Importance
- This shows which variables the model actually relied on most
- Compare this to your earlier correlation / VIF results
"""
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop Features:")
print(importance.head(15))

In [ ]:
# =============================================================================
# Template 2 for a more complex machine learning workflow in Python
# =============================================================================

"""
Notes for grok later:
How might we handle class imbalance? (e.g., SMOTE, class weights)
How might we tune hyperparameters? (e.g., GridSearchCV, RandomizedSearchCV
What do I need to examine to determine class_weight? (e.g., class distribution, ROC-AUC, precision-recall curves)
What is target leakage?
"Censored / Survival Targets" - in real lending some loans haven't matured yet, so we don't know if they will default. How do we handle that?
    - possibly with 'survival analysis' or 'time-to-event' models. 
Default can mean different things across lenders. For example, some lenders consider a loan defaulted after 90 days of non-payment, while others may use 120 days. Make sure to check the definition in your dataset.
How can we create multi-class targets? Maybe number risk 1-5, or use a regression model to predict probability of default.
How do we deal with 'lagged' targets? For example, if we want to predict default in the next 12 months, we need to make sure our features are from before that time period. Otherwise, we might be using future information to predict the past (target leakage).
"""

## Ideas I want to remember later... Or when working on more complex problems:

### Categorical Feature Engineering Strategy: Thematic Grouping + One-Hot Encoding
**Strategy:**  
Instead of blindly one-hot encoding every unique value in a high-cardinality column (like `purpose`), group similar categories together based on business meaning or keyword patterns (e.g., anything related to "car", "truck", "auto", "RV", "motorcycle" → `purpose_vehicle`), then one-hot encode the reduced set of meaningful groups.

- Reduces dimensionality compared to full one-hot encoding.
- Creates more interpretable and business-relevant features.
- Allows me to test specific themes (vehicles, debt consolidation, home improvement, etc.) rather than noisy individual values.

AI says - This is sometimes called **"Semantic Grouping"**, **"Keyword-Based Bucketing"**, or **"Domain-Driven Feature Aggregation"** before encoding.